In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from scraper import fetch_website_links , fetch_website_contents

In [15]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'openai/gpt-oss-120b'
groq = OpenAI(
    api_key = api_key,
    base_url = "https://api.groq.com/v1"
)

API key looks good so far


In [16]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [17]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [18]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [19]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [22]:
def select_relevant_links(url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [24]:
groq = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [25]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [26]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found 7 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'expertise page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'professional profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'professional profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [27]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 9 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'github', 'url': 'https://github.com/huggingface'}]}

In [28]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [29]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash
Updated
2 days ago
•
75.8k
•
1.84k
openbmb/MiniCPM5-2B
Updated
about 2 hours ago
•
67.6k
•
1.21k
XHToken/Spark-X2.5-4B
Updated
9 days ago
•
17.7k
•
1.12k
nex-agi/Nex-N2.5-mini
Updated
4 days ago
•
3.12k
•
700
Qwen/Qwen3.8-27B
Updated
29 days ago

In [30]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [32]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [33]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 5 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash\nUpdated\n2 days ago\n•\n75.8k\n•\n1.84k\nopenbmb/MiniCPM5-2B\nUpdated\nabout 2 hour

In [38]:
def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [39]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 8 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face – The AI Community Building the Future  

---

## 🌟 Company Overview  
Hugging Face is the world’s leading open‑source hub for machine‑learning (ML) collaboration.  With **2 M+ models**, **500 k+ datasets**, and **1 M+ AI applications (Spaces)**, the platform empowers researchers, developers, enterprises, and hobbyists to create, discover, and share state‑of‑the‑art AI tools—all in one unified ecosystem.  

---

## 🚀 Our Mission & Vision  
- **Mission:** Democratize AI by providing a free, transparent, and collaborative platform where anyone can build, share, and deploy machine‑learning models.  
- **Vision:** A vibrant, inclusive community where open‑source AI drives innovation across industry, academia, and society.  

---

## 🛠️ Core Platform Pillars  

| Pillar | What It Is | What You Get |
|--------|------------|--------------|
| **Models** | 2 M+ pre‑trained models covering NLP, vision, speech, multimodal, and more. | Instant access to cutting‑edge models (e.g., DeepSeek‑V4.1‑Flash, MiniCPM5‑2B) with download statistics and community feedback. |
| **Datasets** | 500 k+ curated, version‑controlled datasets (e.g., IMDb reviews, SQuAD, UltraData). | Ready‑to‑use data for training, benchmarking, and research. |
| **Spaces** | A low‑code hosting environment for interactive AI apps (video generation, image editing, chatbots). | Deploy and share demos in seconds—no infrastructure required. |
| **Buckets** | Scalable storage for large model weights, datasets, and artefacts. | Secure, fast, and cost‑effective data management. |
| **Enterprise Solutions** | Hugging Face PRO, Inference Endpoints, Enterprise Support, and custom integration services. | Turnkey production‑grade AI with SLAs, security, and dedicated assistance. |
| **Community Tools** | HuggingChat, Discord, Forum, GitHub, Blog, Daily Papers, and Learn portal. | Continuous learning, real‑time help, and a place to showcase work. |

---

## 🤝 Community & Culture  

- **Open‑source first:** All core libraries (Transformers, Diffusers, Tokenizers, etc.) are community‑driven and freely available.  
- **Collaborative mindset:** Users can **host unlimited public models, datasets, and Spaces**, encouraging peer review and rapid iteration.  
- **Inclusive environment:** Over **100 k AI & ML enthusiasts** follow the organization; active channels include Discord, Forum, and a vibrant GitHub community.  
- **Transparency:** Real‑time activity feeds show contributions, updates, and community milestones.  
- **Innovation at speed:** “Move faster with the HF open‑source stack”—the platform’s tooling accelerates research cycles and product development.  

---

## 🏢 Customers & Partners  

While specific client names aren’t listed on the landing page, Hugging Face serves:

- **Enterprises** seeking secure, scalable AI via PRO plans, Inference Endpoints, and custom support.  
- **Start‑ups & developers** leveraging Spaces to prototype and showcase AI products.  
- **Academic & research institutions** that host datasets and models for reproducible science.  
- **Technology partners** integrating Hugging Face libraries into their own platforms (e.g., cloud providers, AI‑hardware vendors).  

---

## 👩‍💼 Careers & Opportunities  

Hugging Face is a fast‑growing tech company that values:

- **Open‑source contribution** – developers who actively contribute to libraries are a natural fit.  
- **Cross‑functional collaboration** – work with product, research, community, and enterprise teams.  
- **Diversity & inclusion** – a global community with contributors from many backgrounds.  

**Typical roles** (based on public hiring trends):  

- Machine‑Learning Engineer / Research Scientist  
- Software Engineer (backend, frontend, DevOps)  
- Community Manager / Developer Advocate  
- Product Manager – AI Platforms  
- Sales & Enterprise Solutions Engineer  

Interested candidates can **Log In** or **Sign Up** on the website and explore the “Team & Enterprise” section for current openings.  

---

## 📈 Why Join / Partner with Hugging Face?  

- **Scale:** Access to the world’s largest open‑source model repository.  
- **Speed:** Deploy AI instantly with Inference Endpoints or host interactive demos via Spaces.  
- **Support:** Enterprise‑grade SLAs, dedicated support, and professional services.  
- **Impact:** Contribute to a community that shapes the future of AI across industries.  

---

## 📣 Get Started Today  

- **Explore AI Apps** – Browse 1 M+ ready‑to‑run applications.  
- **Browse Models** – Search 2 M+ models and filter by task, language, or popularity.  
- **Join the Community** – Sign up, follow the activity feed, and start collaborating.  

**Website:** https://huggingface.co  
**Discord | Forum | GitHub:** Links available on the navigation bar.  

*Together we’re building the future of AI—one model, dataset, and community member at a time.*

In [43]:
def stream_brochure(company_name, url):
    stream = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [44]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 5 relevant links


**Hugging Face – The AI Community Building the Future**  

---

### 1️⃣ Who We Are  
Hugging Face is the world’s largest open‑source hub for machine‑learning (ML) models, datasets, and interactive applications (Spaces). Our mission is to **democratise AI** by providing a collaborative platform where researchers, developers, and enterprises can create, discover, and share AI resources at scale.

---

### 2️⃣ Core Platform  

| Offering | What It Does | Why It Matters |
|----------|--------------|----------------|
| **Models** | 2 M+ pretrained models (LLMs, vision, audio, multimodal) ready to fine‑tune or deploy. | Accelerates innovation – start from a strong baseline instead of training from scratch. |
| **Datasets** | 500 k+ curated datasets spanning text, code, images, speech, and more. | Guarantees high‑quality data for training, benchmarking, and research. |
| **Spaces** | Hosted, share‑able ML apps (text‑to‑image, video generation, chatbots, etc.) that run on Zero GPU or custom hardware. | Turns prototypes into live demos in seconds; fosters community feedback. |
| **Buckets & Storage** | Scalable object storage for large model artefacts and datasets. | Keeps data and models safe, versioned, and accessible. |
| **Enterprise Solutions** | Hugging Face PRO, Enterprise Support, Inference Endpoints, Inference Providers. | Gives businesses SLA‑backed, secure, and high‑throughput inference pipelines. |
| **Developer Tools** | Docs, API, CLI, GitHub integration, Discord & Forum community, daily papers feed. | Streamlines the end‑to‑end workflow from research to production. |

---

### 3️⃣ Community & Culture  

- **Open‑source at heart** – All public models and datasets are freely available under permissive licences.  
- **Collaboration first** – The Hub enables unlimited public projects; contributors can fork, comment, and improve each other’s work.  
- **Inclusive & Friendly** – A vibrant Discord, forum, and regular blog posts make it easy for newcomers to get help and for veterans to share expertise.  
- **Transparency & Trust** – Every model page shows version history, download counts, and community metrics (likes, forks, discussions).  

> *“Hugging Face is the collaboration platform for the machine learning community.”* – official brand statement  

---

### 4️⃣ Who Uses Hugging Face?  

| Sector | Typical Use Cases |
|--------|-------------------|
| **Tech & SaaS** | Deploy LLM‑powered chatbots, code assistants, recommendation engines. |
| **Research & Academia** | Share state‑of‑the‑art models and benchmark datasets; reproduce papers. |
| **Enterprise & Finance** | Secure, managed inference endpoints for fraud detection, risk modeling. |
| **Creative & Media** | Text‑to‑image/video generation, audio synthesis, interactive art installations. |
| **Healthcare** | Fine‑tune models on clinical notes, radiology images, genomics data (via private buckets). |

Trending public models (e.g., DeepSeek‑V4.1‑Flash, MiniCPM5‑2B, Qwen‑3.8‑27B) illustrate the breadth of high‑impact AI being built on the platform.

---

### 5️⃣ Careers & Growth  

Hugging Face is expanding rapidly and welcomes talent who love **open collaboration, AI research, and product excellence**.  

- **Roles** – Engineering (ML, infrastructure, security), Product, Community & Partnerships, Sales & Enterprise Support.  
- **Benefits** – Remote‑first work, generous learning budget, access to the latest AI models, participation in world‑class research.  
- **How to Apply** – Visit the **Team & Enterprise** section on the website or follow the company on LinkedIn for the latest openings.  

---

### 6️⃣ Get Started  

1. **Explore** – Browse 2 M+ models or 500 k+ datasets directly from the Hub.  
2. **Create** – Spin up a Space in seconds; no GPU required for many demos.  
3. **Collaborate** – Fork, comment, and contribute to any public repo; join the Discord or forum for support.  
4. **Scale** – Upgrade to Hugging Face PRO or Enterprise for private models, SLA‑backed inference, and dedicated support.  

---

### 7️⃣ Contact & Resources  

- **Website**: https://huggingface.co  
- **Docs & API**: https://huggingface.co/docs  
- **Community**: Discord, Forum, GitHub  
- **Brand Assets**: Official logos (SVG/PNG/AI) and colour palette (#FFD21E, #FF9D00, #6B7280) available for partners.  

*Join the world’s most active AI community and shape the future of machine learning together.*

In [45]:


stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 7 relevant links


**Hugging Face – The AI Community Building the Future**  

---

### 🚀 Company Overview  
Hugging Face is the world’s leading collaboration platform for machine‑learning (ML) practitioners. With a vibrant, open‑source ethos, it empowers developers, researchers, and enterprises to **create, discover, and share** models, datasets, and AI applications—all in one unified hub.  

- **2 M+ public models** and **500 k+ datasets** available for instant reuse.  
- **1 M+ AI applications (Spaces)** that showcase interactive demos and production‑ready solutions.  
- A single place to host, version, and deploy ML assets with **unlimited public projects**.  

---

### 🛠️ Core Platform Features  

| Feature | What It Does | Why It Matters |
|---------|--------------|----------------|
| **Models** | Browse, download, and fine‑tune state‑of‑the‑art models (e.g., DeepSeek‑V4.1‑Flash, Qwen‑3.8‑27B). | Accelerates research & product development. |
| **Datasets** | Access curated datasets like IMDb reviews, SQuAD, and UltraData collections. | Reduces data‑gathering time and improves reproducibility. |
| **Spaces** | Deploy interactive demos (image generation, video synthesis, chatbots) with zero‑code or custom code. | Turns prototypes into shareable web apps instantly. |
| **Inference Endpoints & Providers** | Scalable, secure serving of models for production workloads. | Guarantees low‑latency, reliable AI services for enterprises. |
| **Buckets & Storage** | Managed storage for large model files and data artifacts. | Keeps versioned assets organized and accessible. |
| **Enterprise & PRO Plans** | Private repos, dedicated support, compliance tooling, and SLA‑backed uptime. | Meets the security and reliability needs of large organizations. |
| **Community Tools** | Discord, Forum, GitHub, Daily Papers, and a growing ecosystem of plugins. | Fosters knowledge‑sharing and rapid problem‑solving. |

---

### 🌐 Community & Culture  

- **Open‑source at heart** – All public models and datasets are freely reusable under permissive licenses.  
- **Collaboration‑first mindset** – Engineers and researchers co‑author projects, comment on revisions, and contribute to a shared knowledge base.  
- **Transparency & Inclusivity** – Public repos, open discussions on Discord and the Forum, and a commitment to diverse voices in AI.  
- **Innovation Playground** – “Spaces” let anyone experiment with generative AI (text‑to‑image, video, chat) without needing infrastructure.  

The brand’s visual identity (bright HF orange, friendly typography) reflects an approachable, energetic culture that encourages curiosity and sharing.

---

### 🏢 Who Uses Hugging Face?  

| Segment | Typical Users | Example Use Cases |
|---------|---------------|-------------------|
| **Researchers & Academics** | Labs, universities, independent scientists | Benchmarking new models, publishing reproducible results. |
| **Start‑ups & Developers** | AI‑first companies, hobbyists | Rapid prototyping, building SaaS AI features, launching ML‑powered products. |
| **Enterprise Teams** | Fortune‑500 firms, fintech, healthcare, media | Secure model deployment, compliance‑ready pipelines, custom data pipelines. |
| **Educators & Learners** | MOOCs, bootcamps, self‑learners | Hands‑on labs using Spaces, accessing curated datasets for coursework. |

---

### 👩‍💼 Careers & Opportunities  

Hugging Face is constantly expanding its talent pool. Typical roles include:

- **Machine‑Learning Engineers & Researchers** – Build and improve core model libraries, contribute to open‑source projects.  
- **Product & Growth** – Shape the platform experience, drive community adoption, partner with enterprises.  
- **DevOps & Infrastructure** – Scale inference endpoints, maintain storage buckets, ensure platform reliability.  
- **Design & Community** – Craft documentation, run events on Discord/Forum, nurture the global AI community.  

**Why Join?**  
- Work at the intersection of cutting‑edge AI and open‑source collaboration.  
- Influence the tools that power millions of developers worldwide.  
- Enjoy a culture that prizes transparency, learning, and impact.

*Explore current openings on the Hugging Face Careers page (link available on the main site).*

---

### 📣 Get Involved  

- **Explore AI Apps** – Dive into the 1 M+ Spaces to see live demos.  
- **Browse Models & Datasets** – Search the Hub for ready‑to‑use assets.  
- **Join the Conversation** – Participate in Discord, the Forum, or contribute on GitHub.  
- **Start a Project** – Create a free public repo, upload data, and share your work with the world.  

---

### 📞 Contact & Resources  

- **Website:** https://huggingface.co  
- **Docs & API Guides:** Docs → comprehensive tutorials and reference.  
- **Community:** Discord, Forum, Blog, Daily Papers.  
- **Enterprise Inquiries:** Contact the Enterprise & Support teams through the “Enterprise” navigation.  

*Hugging Face – Where the AI community collaborates, innovates, and builds the future together.*